In [1]:
!pip install transformers -q

# 1. Defining encoders

In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as models
from transformers import DistilBertModel, DistilBertConfig

print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

True
Tesla T4


In [3]:
#Image Encoder
class ImageEncoder(nn.Module):
    def __init__(self, embed_dim: int=256, freeze_backbone :bool=True):
        super().__init__()
        resnet=models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)

        self.backbone = nn.Sequential(*list(resnet.children())[:-1])

        if freeze_backbone:
            for param in self.backbone.parameters():
                param.requires_grad=False

        self.projection=nn.Sequential(
            nn.Linear(2048, 512),
            nn.ReLU(),
            nn.Linear(512, embed_dim)
        )

    def forward(self, images: torch.Tensor) -> torch.Tensor:
        features=self.backbone(images)
        features= features.squeeze(-1).squeeze(-1)
        embeddings= self.projection(features)
        return embeddings




In [4]:
#Text encoder
class TextEncoder(nn.Module):
    def __init__(self, embed_dim: int=256, freeze_backbone: bool =True):
        super().__init__()
        self.bert= DistilBertModel.from_pretrained("distilbert-base-uncased")

        if freeze_backbone:
            for param in self.bert.parameters():
                param.requires_grad=False

        hidden_size=self.bert.config.hidden_size
        self.projection=nn.Sequential(
            nn.Linear(hidden_size, 512),
            nn.ReLU(),
            nn.Linear(512, embed_dim),
        )

    def forward(self,input_ids: torch.Tensor, attention_mask: torch.Tensor) -> torch.Tensor:
        outputs=self.bert(input_ids= input_ids, attention_mask=attention_mask)
        cls_outputs= outputs.last_hidden_state[:,0,:]
        embeddings= self.projection(cls_outputs)
        return embeddings


        
        

In [5]:
#InfoNCELoss
class InfoNCELoss(nn.Module):
    def __init__(self, initial_temperature:float=0.07):
        super().__init__()
        self.log_temperature=nn.Parameter(torch.Tensor([initial_temperature]).log())

    @property
    def temperature(self):
        return self.log_temperature.exp().clamp(0.01,100)

    def forward(self, image_embeddings: torch.Tensor, text_embeddings: torch.Tensor) -> torch.Tensor:
        image_embeddings = F.normalize(image_embeddings,dim=-1)
        text_embeddings = F.normalize(text_embeddings,dim=-1)

        logits=(image_embeddings @ text_embeddings.T)/ self.temperature
        batch_size =image_embeddings.size(0)
        labels=torch.arange(batch_size, device=image_embeddings.device)
        loss_i2t= F.cross_entropy(logits, labels)
        loss_t2i= F.cross_entropy(logits.T, labels)
        return (loss_i2t + loss_t2i)/ 2.0


In [6]:
#CLIP Model

class MiniCLIP(nn.Module):
    def __init__(self, embed_dim: int=256, freeze_backbone:bool=True):
        super().__init__()
        self.image_encoder=ImageEncoder(embed_dim=embed_dim, freeze_backbone=freeze_backbone)
        self.text_encoder=TextEncoder(embed_dim=embed_dim, freeze_backbone=freeze_backbone)
        self.loss_fn=InfoNCELoss(initial_temperature=0.07)

    def encode_image(self, images:torch.Tensor) -> torch.Tensor:
        emb=self.image_encoder(images)
        return F.normalize(emb,dim=-1)
    
    def encode_text(self, input_ids: torch.Tensor,attention_mask:torch.Tensor) ->torch.Tensor:
        emb=self.text_encoder(input_ids, attention_mask)
        return F.normalize(emb, dim=-1)

    def forward(self, images, input_ids, attention_mask):
        image_embeddings=self.image_encoder(images)
        text_embeddings=self.text_encoder(input_ids, attention_mask)
        return image_embeddings, text_embeddings
        

In [7]:
#Quick sanity check
if __name__=="__main__":
    device=torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Running on: {device}")

    model=MiniCLIP(embed_dim=256, freeze_backbone=True).to(device)

    B=4
    images= torch.randn(B, 3, 224, 224).to(device)
    input_ids= torch.randint(0, 30522, (B,64)).to(device)
    attn_mask=torch.ones(B, 64,dtype=torch.long).to(device)

    image_emb, text_emb =model(images, input_ids, attn_mask)
    loss=model.loss_fn(image_emb, text_emb)

    print(f"Image embeddings shape: {image_emb.shape}")
    print(f"Text embeddings shape : {text_emb.shape}")
    print(f"InfoNCE loss          : {loss.item():.4f}")
    print(f"Temperature           : {model.loss_fn.temperature.item():.4f}")
    print("Sanity check passed")

Running on: cuda
Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 188MB/s] 


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_transform.weight  | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Image embeddings shape: torch.Size([4, 256])
Text embeddings shape : torch.Size([4, 256])
InfoNCE loss          : 1.3555
Temperature           : 0.0700
Sanity check passed


# 2. Loading dataset

In [8]:
import os
import pandas as pd
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from transformers import DistilBertTokenizer
import torchvision.transforms as T

In [9]:
class Flickr30kDataset(Dataset):
    def __init__(
        self,
        root_dir:str,
        captions_file:str,
        tokenizer: DistilBertTokenizer,
        max_token_len:int=64,
        transform=None,
        one_caption_per_image:bool=False,
        split: str="train",
        val_size: int=1000,
        test_size: int=1000,
        seed:int =42
    ):
        self.root_dir=root_dir
        self.tokenizer=tokenizer
        self.max_token_len=max_token_len
        self.transform= transform or default_transforms(split)

        df=pd.read_csv(captions_file)
        df.columns = df.columns.str.strip()
        df["comment"]=df["comment"].astype(str).str.strip()
        df["image_name"]=df["image_name"].str.strip()

        unique_images=df["image_name"].unique()
        rng=pd.Series(unique_images).sample(frac=1, random_state=seed).values

        test_images=set(rng[:test_size])
        val_images =set(rng[test_size: test_size+val_size])
        train_images=set(rng[test_size+val_size:])

        split_map={"train": train_images,"val":val_images,"test":test_images}
        df=df[df["image_name"].isin(split_map[split])].reset_index(drop=True)

        if one_caption_per_image:
            df=df.groupby("image_name").sample(n=1, random_state=seed).reset_index(drop=True)
            self.df=df
                
    def __len__(self):
         return len(self.df)
    def __getitem__(self,idx):
        row= self.df.iloc[idx]

        #Image
        img_path=os.path.join(self.root_dir,row["image_name"])
        image=Image.open(img_path).convert("RGB")
        image=self.transform(image)

        #Text
        caption=row["comment"]
        encoding=self.tokenizer(
            caption,
            max_length=self.max_token_len,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
        )

        return{
            "image": image,
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask":encoding["attention_mask"].squeeze(0),
            "caption": caption
        }
        

In [10]:
#Transform
def default_transforms(split: str="train"):
    imagenet_mean=[0.485, 0.456,0.406]
    imagenet_std=[0.229, 0.224, 0.225]

    if split=="train":
        return T.Compose([
            T.RandomResizedCrop(224, scale=(0.8,1.0)),
            T.RandomHorizontalFlip(),
            T.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
            T.ToTensor(),
            T.Normalize(mean=imagenet_mean, std=imagenet_std),
        ])
    else:
        return T.Compose([
            T.Resize(256),
            T.CenterCrop(224),
            T.ToTensor(),
            T.Normalize(mean=imagenet_mean, std=imagenet_std),
        ])

In [11]:
#DataLoader factory
def get_dataloaders(
    root_dir: str,
    captions_file: str,
    batch_size: int=64,
    max_token_len: int=64,
    num_workers: int=2,
    val_size: int=1000,
    test_size: int=1000,
):
    tokenizer = DistilBertTokenizer.from_pretrained("distilbert-base-uncased")
    train_ds=Flickr30kDataset(
        root_dir, captions_file, tokenizer,
        max_token_len=max_token_len,
        split="val", val_size=val_size, test_size=test_size,
        one_caption_per_image=True,
    )
    val_ds=Flickr30kDataset(
        root_dir, captions_file, tokenizer,
        max_token_len=max_token_len,
        split="val", val_size=val_size, test_size=test_size,
        one_caption_per_image=True,
    )
    test_ds=Flickr30kDataset(
        root_dir, captions_file, tokenizer,
        max_token_len=max_token_len,
        split="test", val_size=val_size,test_size=test_size,
        one_caption_per_image=True,
    )

    train_loader=DataLoader(
        train_ds, batch_size=batch_size, shuffle=True,
        num_workers=num_workers, pin_memory=True,drop_last=True,
    )
    val_loader=DataLoader(
        val_ds, batch_size=batch_size, shuffle=False,
        num_workers=num_workers, pin_memory=True,
    )
    test_loader=DataLoader(
        test_ds, batch_size=batch_size, shuffle=False,
        num_workers=num_workers, pin_memory=True,
    )

    return train_loader, val_loader, test_loader, tokenizer

In [12]:
if __name__=="__main__":
    ROOT_DIR="/kaggle/input/datasets/eeshawn/flickr30k/flickr30k_images"
    CAPTIONS_FILE="/kaggle/input/datasets/eeshawn/flickr30k/captions.txt"

    train_loader,val_loader, test_loader, tokenizer = get_dataloaders(
        ROOT_DIR, CAPTIONS_FILE, batch_size=4
    )

    batch=next(iter(train_loader))
    print(f"Image shape          : {batch['image'].shape}")
    print(f"input_ids shape      : {batch['input_ids'].shape}")
    print(f"attention_mask       : {batch['attention_mask'].shape}")
    print(f"Sample caption       : {batch['caption'][0]}")
    print(f"Train batches        : {len(train_loader)}")
    print(f"Val batches          : {len(val_loader)}")
    print("Sanity check passed")
    

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Image shape          : torch.Size([4, 3, 224, 224])
input_ids shape      : torch.Size([4, 64])
attention_mask       : torch.Size([4, 64])
Sample caption       : A man sits on his coach and fingers the keys of his electronic keyboard .
Train batches        : 250
Val batches          : 250
Sanity check passed


# 3. Training the model

In [13]:
import os
import torch
import torch.nn as nn
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from tqdm import tqdm

#from model import MiniCLIP
#from dataset import get_loaders

In [14]:
#Config
CONFIG={
    "root_dir":"/kaggle/input/datasets/eeshawn/flickr30k/flickr30k_images",
    "captions_file":"/kaggle/input/datasets/eeshawn/flickr30k/captions.txt",
    "checkpoint_dir":"/kaggle/working/checkpoints",

    "embed_dim": 256,

    "batch_size": 64,
    "num_workers": 2,
    "max_token_len": 64,

    #Phase 1:frozen backbones - higher LR, fewer epochs
    "phase1_epochs":5,
    "phase1_lr": 1e-3,

    #Phase 2: fine-tune everything - lower LR, more epochs
    "phase2_epochs":10,
    "phase2_lr": 1e-5,

    #Misc
    "seed": 42,
}

In [15]:
@torch.no_grad()
def recall_at_k(model, loader, device, ks=(1, 5, 10)):
    model.eval()

    all_image_embs=[]
    all_text_embs=[]

    for batch in tqdm(loader, desc="Computing embeddings", leave=False):
        images =batch['image'].to(device)
        input_ids=batch['input_ids'].to(device)
        attn_mask=batch['attention_mask'].to(device)

        image_emb=model.encode_image(images)
        text_emb=model.encode_text(input_ids, attn_mask)

        all_image_embs.append(image_emb.cpu())
        all_text_embs.append(text_emb.cpu())

    image_embs = torch.cat(all_image_embs, dim=0)
    text_embs= torch.cat(all_text_embs, dim=0)

    sim_matrix=image_embs @ text_embs.T

    N= sim_matrix.size(0)
    results={}

    for k in ks:
        topk_indices=sim_matrix.topk(k, dim=1).indices
        correct = torch.arange(N).unsqueeze(1)
        hits= (topk_indices == correct).any(dim=1).float()
        results[k]= hits.mean().item()* 100
    return results
    
    

In [30]:
#training loop (one epoch)

def train_one_epoch(model, loader, optimizer, device, epoch):
    model.train()
    total_loss =0.0

    pbar= tqdm( loader, desc=f"Epoch {epoch}", leave=True)
    for step, batch in enumerate(pbar):
        images= batch["image"].to(device)
        input_ids=batch["input_ids"].to(device)
        attn_mask=batch["attention_mask"].to(device)

        optimizer.zero_grad()

        image_emb, text_emb = model(images, input_ids, attn_mask)

        loss= model.loss_fn(image_emb, text_emb)
        loss.backward()

        nn.utils. clip_grad_norm(model.parameters(), max_norm=1.0)
        optimizer.step()

        total_loss += loss.item()
        pbar.set_postfix({
            "loss":f"{loss.item():.4f}",
            "tau":f"{model.loss_fn.temperature.item():.4f}",
        })
    return total_loss/ len(loader)

In [26]:
def save_checkpoint(model, optimizer, epoch, val_recall, path):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    torch.save({
        "epoch": epoch,
        "model": model.state_dict(),
        "optimizer": optimizer.state_dict(),
        "val_recall": val_recall,
    },path)
    print(f" Checkpoint saved: {path}")
    

In [27]:
def load_checkpoint(model, optimizer, path, device):
    ckpt= torch.load(path, map_location=device)
    model.load_state_dict(ckpt["model"])
    optimizer.load_state_dict(ckpt["optimizer"])
    print(f" Loaded checkpoint from epoch {ckpt['epoch']}"
          f"(val R@1 = {ckpt['val_recall']:.2f}%)")
    return ckpt["epoch"]

In [28]:
def run_phase(
    model, train_loader, val_loader, device,
    num_epochs, lr, phase_name, checkpoint_dir,
):
    print(f"\n{'='*50}")
    print(f"  {phase_name}")
    print(f"{'='*50}")

    optimizer=AdamW(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=lr,weight_decay=1e-4,
    )
    scheduler =CosineAnnealingLR(optimizer, T_max=num_epochs, eta_min=lr*0.1)
    best_recall=0.0


    for epoch in range(1, num_epochs+1):
        avg_loss=train_one_epoch(model,train_loader, optimizer, device, epoch)

        recalls=recall_at_k(model, val_loader, device)
        scheduler.step()

        print(
            f"   Epoch {epoch:02d} | loss: {avg_loss:.4f} | "
            f" R@1 : {recalls[1]:.1f}% R@5: {recalls[5]:.1f}% R@10 : {recalls[10]:.1f}%"
        )

        if recalls[1] > best_recall:
            best_recall=recalls[1]
            save_checkpoint(
                model, optimizer, epoch, recalls[1],
                path= os.path.join(checkpoint_dir,f"{phase_name.lower().replace(' ','_')}_best.pt"),
            )
    print(f"\n  Best R@1 in {phase_name}: {best_recall:.1f}%")
    return best_recall

In [31]:
#main
def main():
    torch.manual_seed(CONFIG["seed"])
    device=torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Device: {device}")

    train_loader, val_loader, test_loader,_ = get_dataloaders(
        root_dir=CONFIG["root_dir"],
        captions_file=CONFIG["captions_file"],
        batch_size=CONFIG["batch_size"],
        max_token_len=CONFIG["max_token_len"],
        num_workers=CONFIG["num_workers"],
    )
    print(f"Train batches: {len(train_loader)} | Val batches: {len(val_loader)}")
    #----Phase 1:Frozen backbones--------------------
    model=MiniCLIP(embed_dim =CONFIG['embed_dim'],freeze_backbone=True).to(device)

    run_phase(
        model, train_loader, val_loader, device,
        num_epochs= CONFIG["phase1_epochs"],
        lr=CONFIG["phase1_lr"],
        phase_name="Phase 1 - Frozen Backbones",
        checkpoint_dir=CONFIG["checkpoint_dir"],
    )

    #-----Phase 2:Fine-tune everything -----------------------
    #Unfreeze both encoders
    for param in model.image_encoder.backbone.parameters():
        param.requires_grad=True
    for param in model.text_encoder.bert.parameters():
        param.requires_grad=True

    trainable=sum(p.numel() for p in model.parameters() if p.requires_grad)

    run_phase(
        model, train_loader, val_loader, device,
        num_epochs=CONFIG["phase2_epochs"],
        lr=CONFIG["phase2_lr"],
        phase_name="Phase 2- Fine-tuned",
        checkpoint_dir=CONFIG["checkpoint_dir"],
        )
    #------- Final test evaluation ------------------------
    print("\n" + "="*50)
    print(" Final Test Evaluation")
    print("="*50)
    test_recalls = recall_at_k(model, test_loader, device)
    print(
        f"  Test R@1:  {test_recalls[1]:.1f}% "
        f"R@5: {test_recalls[5]:.1f}% "
        f"R@10: {test_recalls[10]:.1f}%"
    )

if __name__=="__main__":
    main()

Device: cuda
Train batches: 15 | Val batches: 16


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_transform.weight  | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



  Phase 1 - Frozen Backbones


Epoch 1:   0%|          | 0/15 [00:00<?, ?it/s]/tmp/ipykernel_58/540334815.py:20: FutureWarning: `torch.nn.utils.clip_grad_norm` is now deprecated in favor of `torch.nn.utils.clip_grad_norm_`.
  nn.utils. clip_grad_norm(model.parameters(), max_norm=1.0)
Epoch 1: 100%|██████████| 15/15 [00:08<00:00,  1.70it/s, loss=3.8440, tau=0.0702]


...

Epoch 9: 100%|██████████| 15/15 [00:15<00:00,  1.00s/it, loss=0.6979, tau=0.0698]


   Epoch 09 | loss: 0.6880 |  R@1 : 69.6% R@5: 94.6% R@10 : 98.1%
 Checkpoint saved: /kaggle/working/checkpoints/phase_2-_fine-tuned_best.pt


Epoch 10: 100%|██████████| 15/15 [00:15<00:00,  1.00s/it, loss=0.6738, tau=0.0698]


   Epoch 10 | loss: 0.6781 |  R@1 : 72.1% R@5: 95.0% R@10 : 98.4%
 Checkpoint saved: /kaggle/working/checkpoints/phase_2-_fine-tuned_best.pt

  Best R@1 in Phase 2- Fine-tuned: 72.1%

 Final Test Evaluation


  Test R@1:  3.0% R@5: 12.9% R@10: 19.4%
